# T04 · The theory behind T05: what E/F training leaves free, PHL's Hessian loss and its estimator, and what it costs

**openQHA theory notebook — the version of 2026-09-22 (PHL verbatim).** It rests on the choices
behind the run: the training target is the Cartesian Hessian matrix, sampled by Hessian–vector
products (PHL's loss as published); basin frames only (the held-out generator is what the judge
reads); validation by four probes fixed per frame; the Replay as a drawn file (round 1: a
30,000-frame draw in two arms, `config_weight` 1 / 10, the mace-docs multihead guidance; user
ruling 2026-09-30); the judge's gate on the matrix itself, with frequencies and the entropy
computed from it afterwards. It replaces the version of the same day that still
carried the projected, mass-weighted, entropy-weighted variants of the loss (T03's design,
since superseded) as "diagnostics", together with the rigid-block algebra, the band table
and the cubic constants that only those variants needed. **Nothing in the training is projected
except the random probe.** Frequencies are read from a trained matrix by the standard vibrational
analysis (mass-weighting + Eckart projection, `hessian_compare`) — an evaluation step, exactly as
HIP §5 does it, never a term of the loss.

T05 states the goal — fine-tune MACE-OFF23_medium with PHL's loss so that its energies, forces
**and Hessians** reach ωB97M-D3(BJ)/def2-TZVPPD at the molecules' minima — and gives the recipe.
This notebook is the derivation underneath it. Every statement is derived from the Taylor
expansion or from Hutchinson's identity, and every identity is checked in the cell that follows
it on

* `tests/data/propanal_molecule/`: the ωB97M-D3(BJ)/def2-TZVPPD analytic Hessian and the stored
  MACE-OFF23_medium Hessian at the same basin geometry;
* `tests/data/methyloxirane_frames/`: the Frame set of 2-methyloxirane (QM9 000044) — the basin
  and its four classical-298 K displaced frames, each with **energy, forces and the analytic
  Hessian at both levels at the same positions**: in the campaign such displaced frames are
  held-out generator frames — E/F labels, never trained on, read by the judge as
  reference rows; here they have Hessians too, so the reading can be shown before training;
* MACE-OFF23_medium itself, when it loads (the cells say "skipped" otherwise).

Reading order: §1 what E/F training leaves free (the curvature at and between samples, and what
a displaced force does constrain) → §2 PHL's loss and its estimator: unbiased, its variance, exact
with $3N$ unit probes, and what it bounds (every frequency, by Weyl) → §3 the surface off the
minimum: third derivatives and what the held-out generator reads → §4 the Hessian–vector product,
its gradient and its cost, measured → §5 the total objective: basin frames labelled, Replay frames
unlabelled → §6 complete algorithm listings, as the code stands → §7 acceptance, the code map,
and how T05 uses each piece.

Conventions: $N$ atoms, $x\in\mathbb R^{3N}$ (Å), $E$ in eV, $g=\nabla E=-F$, $H=\nabla^2E$
(eV Å$^{-2}$), $\Delta H=H_\theta-H_r$; $\|\cdot\|_F$ Frobenius, $\|\cdot\|_2$ spectral; $\theta$ the
model parameters, subscript $r$ the reference.

In [ ]:
import sys, math, time
from pathlib import Path
import numpy as np
import torch
torch.set_default_dtype(torch.float64)

def _repo_root():
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if (p / "openqha" / "__init__.py").is_file():
            return p
    raise RuntimeError("run from inside the openQHA checkout")

ROOT = _repo_root(); sys.path.insert(0, str(ROOT))
from ase.data import atomic_masses, atomic_numbers
from openqha.qm_interfaces import orca
from openqha.thermochem import hessian as hessian_mod, hessian_compare as hc
from openqha.data import frames as frames_mod
from openqha_hessian import phl, hvp as hvp_mod

LEVEL = "wb97m-d3bj_def2-tzvppd"; MLEVEL = "mace-off23_medium"
to_cm = lambda lam: np.sign(lam) * np.sqrt(np.abs(lam)) * hessian_mod.CM_INV_PER_SQRT_EV_A2_AMU
masses_of = lambda atoms: np.array([atomic_masses[atomic_numbers[s]] for s in atoms.get_chemical_symbols()])

# --- propanal, basin 0: reference H_r and MACE H_t at the wB97M minimum
FIXP = ROOT / "tests" / "data" / "propanal_molecule"
parsed = orca.parse_hess(FIXP / "msrrho" / f"orca.{LEVEL}.basin00.hess")
sym_p = list(parsed["symbols"])
xp = np.asarray(parsed["positions_bohr"]) / orca.BOHR_PER_ANGSTROM
mp = np.array([atomic_masses[atomic_numbers[s]] for s in sym_p])
Hp_r = orca.hessian_to_ev_per_angstrom2(parsed["hessian_eh_bohr2"])
Hp_t = np.load(FIXP / "mace" / "basin00" / f"hessian_at_{LEVEL}.npy")
Np = len(sym_p)

# --- 2-methyloxirane: basin + 4 displaced frames, E/F/H at both levels, identical positions
FIXM = ROOT / "tests" / "data" / "methyloxirane_frames"
fr_t = frames_mod.read_frames(FIXM / f"basin.{MLEVEL}.extxyz") + frames_mod.read_frames(FIXM / f"displaced.{MLEVEL}.extxyz")
fr_r = frames_mod.read_frames(FIXM / f"basin.{LEVEL}.extxyz") + frames_mod.read_frames(FIXM / f"displaced.{LEVEL}.extxyz")
assert all(np.abs(a.positions - b.positions).max() < 1e-7 for a, b in zip(fr_t, fr_r))
Nm = len(fr_t[0]); n3m = 3 * Nm
mm = masses_of(fr_t[0])
X = [a.positions.copy() for a in fr_r]                                   # x_0 (basin), x_1..x_4 (displaced)
E_r = [a.get_potential_energy() for a in fr_r]; E_t = [a.get_potential_energy() for a in fr_t]
G_r = [-a.get_forces().reshape(-1) for a in fr_r]; G_t = [-a.get_forces().reshape(-1) for a in fr_t]   # gradients g = -F
H_r = [np.asarray(a.info["hessian"], float).reshape(n3m, n3m) for a in fr_r]
H_t = [np.asarray(a.info["hessian"], float).reshape(n3m, n3m) for a in fr_t]
rms = [a.info["rms_displacement_A"] for a in fr_t]
print(f"propanal: N = {Np}; 2-methyloxirane ({fr_t[0].info['smiles']}): N = {Nm}, {len(fr_t)} frames, RMS displacements {np.round(rms, 3)} Å")
print("frame   E_r-E_0 (kcal/mol)   |g_r|max (eV/Å)   |g_t|max   |H_r-H_r^T|max   |H_t-H_t^T|max")
for k in range(len(fr_r)):
    print(f"  {k}      {(E_r[k]-E_r[0])*23.0605:8.2f}            {np.abs(G_r[k]).max():6.3f}         {np.abs(G_t[k]).max():6.3f}       {np.abs(H_r[k]-H_r[k].T).max():.1e}          {np.abs(H_t[k]-H_t[k].T).max():.1e}")

propanal: N = 10; 2-methyloxirane (C[C@H]1CO1): N = 10, 5 frames, RMS displacements [0.    0.058 0.096 0.127 0.059] Å
frame   E_r-E_0 (kcal/mol)   |g_r|max (eV/Å)   |g_t|max   |H_r-H_r^T|max   |H_t-H_t^T|max
  0          0.00             0.067          0.000       0.0e+00          0.0e+00
  1          7.25             1.734          1.705       0.0e+00          0.0e+00
  2          4.33             1.039          1.054       0.0e+00          0.0e+00
  3          8.87             2.077          2.130       0.0e+00          0.0e+00
  4          4.90             2.039          1.981       0.0e+00          0.0e+00


## 1. What energy–force training leaves free: the curvature at and between samples

**Proposition 1 (the free function).** Let $\{x_n\}$ be any finite set of training geometries
with pairwise distance $\ge 2\rho>0$, and $\{S_n\}$ any symmetric matrices. There is a smooth
$\varphi$ with
$$\varphi(x_n)=0,\qquad \nabla\varphi(x_n)=0,\qquad \nabla^2\varphi(x_n)=S_n\quad\text{for every }n .$$
*Proof.* Take a bump $b\in C^\infty(\mathbb R)$ with $b(t)=1$ for $|t|\le\tfrac12$, $b(t)=0$ for
$|t|\ge1$, and set
$$\varphi(x)=\sum_n \tfrac12\,(x-x_n)^{\!\top}S_n(x-x_n)\;b\!\left(\frac{\|x-x_n\|}{\rho}\right).$$
The supports are disjoint, so near $x_n$ only term $n$ is present and there $b\equiv1$: $\varphi$
is the quadratic $\tfrac12 d^{\!\top}S_nd$, $d=x-x_n$, whose value and gradient vanish at $d=0$
and whose Hessian is $S_n$. $\square$

So $\tilde E=E+\varphi$ has **the same energies and the same forces as $E$ at every training
geometry** and an arbitrary Hessian there. An E/F loss cannot tell $E$ from $\tilde E$; whatever
curvature the fitted model has at (and between) the samples is decided by the model class and
the optimiser, not by the data. That is the mechanism behind two facts T05 rests on: MACE-OFF23
was trained on 951,813 E/F frames and its Hessians were never constrained (T05 §1 measures
what that left: 0.07–0.08 eV Å$^{-2}$ per element, 5–6 cm$^{-1}$ per mode); and PFT's observation
that E/F fine-tuning on displaced frames improved forces and *degraded* phonons. It is the
reason a Hessian term is in the loss at all. The proposition holds at every sample, stationary
or not: PHL labels every frame with $H$; we label the basins only, and §3 says what
that leaves to the model and how the judge reads it.

**What a displaced force does constrain.** The forces at two points are not independent of
the Hessian between them. By the fundamental theorem of calculus along the segment
$x_0+t\,d$, $t\in[0,1]$:
$$g(x_0+d)-g(x_0)=\int_0^1 H(x_0+t\,d)\,d\;dt\;=\;\bar H_d\,d ,\qquad
\bar H_d:=\int_0^1H(x_0+td)\,dt .\tag{1.1}$$
A pair (basin, displaced frame) with forces at both therefore fixes **one column-direction of
the path-averaged Hessian**, $\bar H_d\,d$: $3N$ numbers per frame, out of the $\sim\tfrac12(3N)^2$
of $H$. With the classical 298 K draw the displacement is a sum over all modes, so (1.1)
mixes every mode's curvature into one vector; four frames per basin give four such vectors.
The trapezoid $\bar H_d\approx\tfrac12(H(x_0)+H(x_0+d))$ is exact to $O(\|d\|^3)$ (Simpson's
error term with the third derivative). The cell measures both on the reference labels: how
far the basin Hessian alone is from the force difference (the size of the change of curvature
along $d$) and how far the trapezoid is (the third-derivative remainder). The last column is
the relative change of the reference Hessian itself between the basin and the frame.

In [2]:
# Proposition 1 in one dimension, numerically: E(x) = cos x on samples x_n; E~ = E + phi with S_n = +2, -3, +5
b = lambda t: np.where(np.abs(t) < 1, np.exp(1 - 1 / np.maximum(1 - t ** 2, 1e-300)), 0.0)   # smooth bump, b(0)=1
xs = np.array([0.5, 2.0, 3.5]); S = np.array([2.0, -3.0, 5.0]); rho = 0.6
E = lambda x: np.cos(x)
phi = lambda x: sum(0.5 * s * (x - xn) ** 2 * b((x - xn) / rho) for xn, s in zip(xs, S))
h = 1e-4
d1 = lambda f, x: (f(x + h) - f(x - h)) / (2 * h); d2 = lambda f, x: (f(x + h) - 2 * f(x) + f(x - h)) / h ** 2
print("at the samples:  E~-E   F~-F        H~-H     S_n")
for xn, s in zip(xs, S):
    print(f"   x={xn:3.1f}      {phi(xn):.1e}  {d1(phi, xn):.1e}   {d2(phi, xn):+.6f}  {s:+.1f}")

# (1.1) on the reference labels of 2-methyloxirane: g(x_k) - g(x_0) vs the trapezoid and vs H(x_0) d alone
print("\n(1.1) reference level, per displaced frame k:  |dg|   |dg - H0 d| / |dg|   |dg - ½(H0+Hk) d| / |dg|   ‖H_r(x_k)-H_r(x_0)‖/‖H_r(x_0)‖")
for k in range(1, 5):
    d = (X[k] - X[0]).reshape(-1); dg = G_r[k] - G_r[0]
    e0 = np.linalg.norm(dg - H_r[0] @ d) / np.linalg.norm(dg)
    e1 = np.linalg.norm(dg - 0.5 * (H_r[0] + H_r[k]) @ d) / np.linalg.norm(dg)
    dH = np.linalg.norm(H_r[k] - H_r[0]) / np.linalg.norm(H_r[0])
    print(f"   k={k}  RMS {rms[k]:.3f} Å:   {np.linalg.norm(dg):6.3f}      {e0:.3f}                {e1:.4f}                    {dH:.3f}")
print("-> at 0.06-0.13 Å RMS (a classical 298 K frame) the reference Hessian has changed by 15-22 %, and the basin Hessian alone")
print("   misses the force difference by 20-90 %; the trapezoid (two Hessians) gets it to 1-19 %, the remainder being the third")
print("   derivative along d. A label set with Hessians at the basins only says nothing about this change: §3 measures what the")
print("   held-out generator frames read off the base model before any training (T05 §3).")

at the samples:  E~-E   F~-F        H~-H     S_n
   x=0.5      0.0e+00  0.0e+00   +2.000000  +2.0
   x=2.0      0.0e+00  -3.3e-16   -3.000000  -3.0
   x=3.5      0.0e+00  0.0e+00   +5.000000  +5.0

(1.1) reference level, per displaced frame k:  |dg|   |dg - H0 d| / |dg|   |dg - ½(H0+Hk) d| / |dg|   ‖H_r(x_k)-H_r(x_0)‖/‖H_r(x_0)‖
   k=1  RMS 0.058 Å:    4.018      0.190                0.0135                    0.149
   k=2  RMS 0.096 Å:    2.537      0.905                0.1285                    0.152
   k=3  RMS 0.127 Å:    4.695      0.749                0.1892                    0.223
   k=4  RMS 0.059 Å:    4.129      0.189                0.0145                    0.164
-> at 0.06-0.13 Å RMS (a classical 298 K frame) the reference Hessian has changed by 15-22 %, and the basin Hessian alone
   misses the force difference by 20-90 %; the trapezoid (two Hessians) gets it to 1-19 %, the remainder being the third
   derivative along d. A label set with Hessians at the basins only says n

## 2. PHL's loss and its estimator: unbiased, its variance, exact with $3N$ unit probes

**The loss (PHL eq. 6 and Algorithm 1).** For a labelled frame the Hessian term is the mean squared
error of the Cartesian matrix as stored,
$$\mathcal L_H=\frac{1}{(3N)^2}\,\|H_\theta-H_r\|_F^2 ,\tag{2.1}$$
and PHL never forms $H_\theta$: with $K$ random probes $v_j$ whose components are independent with zero
mean and unit variance ($v_j\sim\mathcal N(0,I)$ in Algorithm 1, $\pm1$ Rademacher in Hutchinson's
original), it trains on
$$\hat{\mathcal L}_H^{(K)}=\frac{1}{9N^2K}\sum_{j=1}^K\big\|H_\theta v_j-H_rv_j\big\|^2 ,\qquad
H_\theta v_j=-\nabla_x(F_\theta\!\cdot v_j)\ \text{(one extra backward, §4)},\quad H_rv_j\ \text{a matvec on the Label} .\tag{2.2}$$
This is the whole training rule (`phl_loss`; the code's `probe = rademacher | gaussian`, `n_probes = K`).

**Unbiased.** For $E[v]=0$, $E[vv^{\!\top}]=I$,
$E\|\Delta Hv\|^2=\operatorname{tr}(\Delta H^{\!\top}\Delta H\,E[vv^{\!\top}])=\|\Delta H\|_F^2$, so
$E[\hat{\mathcal L}_H^{(K)}]=\mathcal L_H$ for every $K$ — Hutchinson's identity with the matrix
$\Delta H$ in place of the trace. The gradient with respect to $\theta$ is unbiased for the same reason
(expectation and $\partial_\theta$ commute; the probes do not depend on $\theta$).

**Variance.** For Rademacher probes, with $A=\Delta H^{\!\top}\Delta H$,
$$\operatorname{Var}\big[\hat{\mathcal L}_H^{(K)}\big]=\frac{2}{(9N^2)^2K}\Big(\|A\|_F^2-\sum_iA_{ii}^2\Big) ,\tag{2.3}$$
(T03 eq. 8, `phl.estimator_variance`); Gaussian probes have the larger variance
$2\|A\|_F^2/((9N^2)^2K)$ at the same mean; Rademacher has the smaller variance, and the code's default draw is PHL's standard normal.

**Exact with $3N$ unit probes.** For any orthonormal $U$ with $UU^{\!\top}=I$,
$\sum_j\|\Delta HU_j\|^2=\|\Delta H\|_F^2$ with zero variance. With $U=I$ the $3N$ probes $e_j$ give
(2.1) exactly, and $H_\theta e_j$ *is* column $j$ of `get_hessian` (§4 checks this to the last digit):
the full matrix is the deterministic limit of the same estimator at $3N$ times the cost of one
probe. The validation term is (2.2) with the frame's STORED $K=4$ standard-normal probes fixed per frame — the
same estimator, reproducible, at ~4/3N of the full matrix.

**What (2.1) bounds.** Frequencies are eigenvalues of the mass-weighted, Eckart-projected matrix
(the standard analysis, an evaluation step). By Weyl's inequality, for $K=M^{-1/2}HM^{-1/2}$,
$$|\lambda_i(K_\theta)-\lambda_i(K_r)|\;\le\;\|\Delta K\|_2\;\le\;\|\Delta K\|_F\;\le\;\|\Delta H\|_F/m_{\min} ,\tag{2.4}$$
so driving (2.1) to zero drives every frequency to the reference's — the Cartesian target is
*sufficient* for every frequency. The bound is loose (it does not know which modes are soft), which
is why the judge measures the low-mode line rather than infers it. The cell checks (2.2)–(2.4) on the
propanal base model: 4000 draws of both probe kinds, the $3N$ unit probes, and the Weyl chain.

In [3]:
rng = np.random.default_rng(0)
dH = Hp_t - Hp_r; n3p = 3 * Np; nu = 9 * Np * Np
fro2 = np.sum(dH ** 2)
def draws(gaussian, n=4000):
    V = rng.standard_normal((n, n3p)) if gaussian else rng.choice([-1.0, 1.0], size=(n, n3p))
    return np.sum((V @ dH.T) ** 2, axis=1)
A = dH.T @ dH; var_rad = 2 * (np.sum(A ** 2) - np.sum(np.diag(A) ** 2)); var_gau = 2 * np.sum(A ** 2)
d_r, d_g = draws(False), draws(True)
print(f"propanal, base model vs reference, (2.1):  ‖ΔH‖²_F/(9N²) = {fro2/nu:.6e}   (N = {Np}, 3N = {n3p})")
print(f"  Rademacher, 4000 single-probe draws: mean {d_r.mean()/nu:.6e} ({d_r.mean()/fro2-1:+.2%}), var {d_r.var():.3e} vs (2.3) {var_rad:.3e}")
print(f"  Gaussian,   4000 single-probe draws: mean {d_g.mean()/nu:.6e} ({d_g.mean()/fro2-1:+.2%}), var {d_g.var():.3e} vs 2‖A‖²_F {var_gau:.3e}  ({var_gau/var_rad:.2f}x Rademacher)")
print(f"  3N = {n3p} unit probes: {np.sum((dH @ np.eye(n3p)) ** 2)/nu:.6e}  (exact, zero variance)")
# the code's probe path: phl.make_probes on the Cartesian target, k = 4, and the s.e. it predicts
vt, r, info = phl.make_probes(Hp_r, mode="rademacher", k=4, rng=rng)
est = phl.estimator_from_products(np.asarray([Hp_t @ v for v in vt]), r, info)
se = np.sqrt(phl.estimator_variance(Hp_t, Hp_r, k=4)["rademacher"])
print(f"  phl.make_probes(k=4): one draw {est:.6e}, s.e. from (2.3) {se:.1e} -> {abs(est - fro2/nu)/se:.2f} s.e. from the exact value;"
      f" denominator {info['denominator']} = 9N²k {nu*4}")
# (2.4) Weyl on the mass-weighted matrices (no projection needed for the bound; frequencies are the projected eigenvalues)
m3p = np.repeat(mp, 3)
Kt = Hp_t / np.sqrt(np.outer(m3p, m3p)); Kr = Hp_r / np.sqrt(np.outer(m3p, m3p)); dK = Kt - Kr
lt, lr = np.linalg.eigvalsh(Kt), np.linalg.eigvalsh(Kr)
print(f"\n(2.4) Weyl: max_i |λ_i(K_θ) - λ_i(K_r)| = {np.abs(lt - lr).max():.3e}  ≤  ‖ΔK‖₂ = {np.linalg.norm(dK, 2):.3e}  ≤  ‖ΔK‖_F = {np.linalg.norm(dK):.3e}"
      f"  ≤  ‖ΔH‖_F / m_min = {np.linalg.norm(dH)/mp.min():.3e}   [eV Å⁻² amu⁻¹]")
print("-> (2.2) is unbiased for both probe kinds and Rademacher has the smaller variance; the 3N unit probes are exact and equal")
print("   get_hessian (§4). The Weyl chain holds and is loose by orders of magnitude on the soft modes: the target bounds every")
print("   frequency, the judge still measures them.")

propanal, base model vs reference, (2.1):  ‖ΔH‖²_F/(9N²) = 2.590998e-03   (N = 10, 3N = 30)
  Rademacher, 4000 single-probe draws: mean 2.638488e-03 (+1.83%), var 1.407e+00 vs (2.3) 1.321e+00
  Gaussian,   4000 single-probe draws: mean 2.597506e-03 (+0.25%), var 2.377e+00 vs 2‖A‖²_F 2.322e+00  (1.76x Rademacher)
  3N = 30 unit probes: 2.590998e-03  (exact, zero variance)
  phl.make_probes(k=4): one draw 2.749449e-03, s.e. from (2.3) 6.4e-04 -> 0.25 s.e. from the exact value; denominator 3600 = 9N²k 3600

(2.4) Weyl: max_i |λ_i(K_θ) - λ_i(K_r)| = 3.427e-01  ≤  ‖ΔK‖₂ = 3.697e-01  ≤  ‖ΔK‖_F = 5.449e-01  ≤  ‖ΔH‖_F / m_min = 1.515e+00   [eV Å⁻² amu⁻¹]
-> (2.2) is unbiased for both probe kinds and Rademacher has the smaller variance; the 3N unit probes are exact and equal
   get_hessian (§4). The Weyl chain holds and is loose by orders of magnitude on the soft modes: the target bounds every
   frequency, the judge still measures them.


## 3. The surface off the minimum: third derivatives, and what the held-out generator reads

**Taylor expansion of the Hessian field.** Along a displacement $d$ from a basin $x_0$,
$$H(x_0+d)=H(x_0)+\mathcal T[d]+O(\|d\|^2),\qquad \mathcal T[d]_{ij}=\sum_k\frac{\partial^3E}{\partial x_i\partial x_j\partial x_k}\,d_k .\tag{3.1}$$
A Hessian label at $x_0$ and one at $x_0+d$ together measure one slice $\mathcal T[d]$ of the
third-derivative tensor; a pair of *models* does the same, so the model's anharmonicity along $d$ can
be compared with the reference's:
$$\Delta\mathcal T[d]\;\approx\;\big(H_\theta(x_0+d)-H_\theta(x_0)\big)-\big(H_r(x_0+d)-H_r(x_0)\big) .\tag{3.2}$$

**What the held-out generator reads (T05 §3).** By Proposition 1 an E/F label at $x_0+d$
constrains nothing about $H(x_0+d)$ beyond (1.1)'s one direction; by (3.1) the reference Hessian
there differs from the basin's by a whole slice of the cubic tensor. The campaign trains on basin
Hessians only (`dataset.build(train_generators=("basin",))`): the slice $\mathcal T[d]$ is
left to the model class *by decision*, because the deliverable — the msRRHO entropy at the minima —
reads $H(x_0)$ alone, and Rodriguez 2025 measured that stationary-only Hessian training still cut the
off-stationary Hessian error by 70–90 %. The displaced, merged and saddle frames are held-out
generator frames: labelled E/F (here also H), never in `train` or `valid`, read by the judge in
RMS-displacement bins as *reference* rows. The number those rows read is the target's own error at
a displaced frame against the same error at the basin,
$$\rho_k=\frac{\|\Delta H(x_k)\|_F^2}{\|\Delta H(x_0)\|_F^2} ,\tag{3.3}$$
(`judge.aggregate_displacement`, engine against base per bin). $\rho_k\gg1$ is the PFT symptom
(curvature roughened between samples); after the fine-tune it is reported beside the gate rows and
triggers nothing (no second stage, no displaced Hessians bought). The cell applies (3.2)–(3.3) to the
2-methyloxirane frames at both levels for the base model.

In [ ]:
print("2-methyloxirane, base model vs reference (Hessians at both levels at the same frames):")
print("frame  RMS(Å)   ‖ΔH‖²/(9N²) (2.1)    ρ_k (3.3)   ‖H_r(x_k)-H_r(x_0)‖/‖H_r(x_0)‖   |ΔT[d]|/|T_r[d]| (3.2)   generator, who reads it")
c0 = np.linalg.norm(H_t[0] - H_r[0]) ** 2 / (9 * Nm * Nm)
for k in range(5):
    cart = np.linalg.norm(H_t[k] - H_r[k]) ** 2 / (9 * Nm * Nm)
    if k:
        Tr = H_r[k] - H_r[0]; Tt = H_t[k] - H_t[0]
        dHr, mis = np.linalg.norm(Tr) / np.linalg.norm(H_r[0]), np.linalg.norm(Tt - Tr) / np.linalg.norm(Tr)
    gen = fr_t[k].info["generator"]
    print(f"  {k}    {rms[k]:.3f}    {cart:.5f}            {cart/c0:5.2f}          {'-' if k == 0 else f'{dHr:.3f}':>8s}                        {'-' if k == 0 else f'{mis:.2f}':>6s}             {gen}: {'train / valid (the loss)' if gen == 'basin' else 'test, a reference bin (the judge)'}")
print("-> over a 0.06-0.13 Å RMS frame the reference Hessian itself changes by 15-22 % and the base model reproduces that change to")
print("   a few per cent (3.2); its target error there is ρ_k times the basin's. The fine-tune sees frame 0 only; frames 1-4")
print("   are held-out generator frames whose ρ_k the judge reads in the '<0.08' / '<0.15' Å bins, engine against base -- reported, never gated.")

2-methyloxirane, base model vs reference (Hessians at both levels at the same frames):
frame  RMS(Å)   ‖ΔH‖²/(9N²) (2.1)    ρ_k (3.3)   ‖H_r(x_k)-H_r(x_0)‖/‖H_r(x_0)‖   |ΔT[d]|/|T_r[d]| (3.2)   generator, who reads it
  0    0.000    0.02869             1.00                 -                             -             basin: train / valid (the loss)
  1    0.058    0.02298             0.80             0.149                          0.08             displaced: test, a reference bin (the judge)
  2    0.096    0.01728             0.60             0.152                          0.07             displaced: test, a reference bin (the judge)
  3    0.127    0.02709             0.94             0.223                          0.04             displaced: test, a reference bin (the judge)
  4    0.059    0.02648             0.92             0.164                          0.07             displaced: test, a reference bin (the judge)
-> over a 0.06-0.13 Å RMS frame the reference Hessian itself chan

## 4. The Hessian–vector product: gradient and cost, derived and measured

**The product.** With $F_\theta=-\nabla_xE_\theta$ built with a graph (`create_graph=True`), for a
constant probe $v$,
$$H_\theta v=\nabla_x\big(\nabla_xE_\theta\cdot v\big)=-\nabla_x\big(F_\theta\cdot v\big) ,\tag{4.1}$$
one more backward pass over the force graph (T03 eq. 9; `hvp.hvp_from_forces`; PHL's
`torch.autograd.grad(forces, coordinates, grad_outputs=vectors, create_graph=True)` is the same
line). Nothing in MACE's forward changes: in training mode the forces already carry the graph
and `batch.positions` is the leaf they were taken against.

**The gradient of the estimator.** Let $\rho_j=H_\theta v_j-r_j$ with $r_j=H_rv_j$ a constant. Then
$$\frac{\partial\hat{\mathcal L}_H^{(K)}}{\partial\theta}=\frac2{9N^2K}\sum_j\rho_j^{\!\top}\,\frac{\partial(H_\theta v_j)}{\partial\theta},
\qquad \frac{\partial(H_\theta v_j)}{\partial\theta}=-\frac{\partial}{\partial\theta}\nabla_x\big(F_\theta\cdot v_j\big)
=-\frac{\partial^3E_\theta}{\partial\theta\,\partial x\,\partial x}v_j ,\tag{4.2}$$
a mixed third derivative that autograd delivers when the HVP itself is built with
`create_graph=True` (the "third-order graph"; PFT's "triple backward"). The E/F terms need
only $\partial^2E/\partial\theta\partial x$.

**Cost model.** Reverse-mode AD of a scalar costs a fixed small multiple of the forward it
differentiates (Baur–Strassen). Take one *energy + forces with graph* pass as the unit $u$
(a forward plus one backward, the graph kept). One HVP is one more backward over that
graph — a graph of about the same size — so each probe costs $\approx1\,u$, and the probes
share the force graph: **the Hessian term of a training step costs $\approx k\,u$ on top of
the $1\,u$ the E/F terms already pay**, plus the parameter backward through it. The full
matrix by `vmap` is the same product for the $3N$ unit vectors, $\approx3N\,u$ (vmap batches the
backward passes but does not shorten them). So the ratio full : $k$-probe is $\approx3N:k$ —
for the campaign's 19-atom molecules and $k=4$, $57:4$ per step — and the full-matrix memory
grows with the $3N$ retained graphs. The cell measures the actual wall time on
MACE-OFF23_medium at $N=10$: $u$, then $k=1,4,8,30$ HVPs on the same force graph, then
`get_hessian`, and checks that the $3N$ unit-probe HVPs *are* `get_hessian` — which is why
the full matrix (the judge's tool, §2) costs $3N\,u$ per frame and no more, and the fixed-probe
validation $4\,u$.

In [5]:
try:
    from ase import Atoms
    from openqha.potentials import engine
    calc, ename, prov = engine.calculator(device="cpu")
    atoms0 = Atoms(symbols=sym_p, positions=xp)
    model = calc.models[0]
    def timed(fn, reps=3):
        ts = []
        for _ in range(reps):
            t0 = time.perf_counter(); out = fn(); ts.append(time.perf_counter() - t0)
        return min(ts), out
    b = calc._atoms_to_batch(atoms0)
    def ef():
        bd = calc._clone_batch(b).to_dict()
        out = model(bd, training=True, compute_force=True, compute_stress=False)
        return out["forces"], bd["positions"]
    t_ef, (F_m, pos) = timed(ef)
    rows = []
    for k in (1, 4, 8, 30):
        probes = torch.as_tensor(rng.choice([-1.0, 1.0], size=(k, Np, 3)), dtype=F_m.dtype)
        t_k, _ = timed(lambda: hvp_mod.hvp_from_forces(F_m, pos, probes, create_graph=False))
        rows.append((k, t_k))
    t_full, Hm = timed(lambda: np.asarray(calc.get_hessian(atoms0)).reshape(3 * Np, 3 * Np))
    print(f"{ename}, N = {Np} (3N = {3*Np}), CPU wall (best of 3):")
    print(f"  energy + forces with graph         {t_ef*1e3:7.1f} ms   (= 1 unit)")
    for k, t_k in rows:
        print(f"  {k:2d} HVP on that graph               {t_k*1e3:7.1f} ms   {t_k/t_ef:5.2f} units   {t_k/t_ef/k:5.2f} per probe")
    print(f"  get_hessian (vmap over 3N = {3*Np})    {t_full*1e3:7.1f} ms   {t_full/t_ef:5.2f} units")
    hv30 = hvp_mod.hvp_from_forces(F_m, pos, torch.eye(3 * Np, dtype=F_m.dtype).reshape(3 * Np, Np, 3)).detach().numpy().reshape(3 * Np, 3 * Np)
    print(f"  3N unit-probe HVPs == get_hessian to {np.abs(hv30 - Hm).max():.1e} eV/Å²;  stored fixture Hessian to {np.abs(Hm - Hp_t).max():.1e}")
    # (2.2) on the live model: k = 4 Rademacher probes against the exact (2.1)
    probes4 = torch.as_tensor(rng.choice([-1.0, 1.0], size=(4, Np, 3)), dtype=F_m.dtype)
    hv4 = hvp_mod.hvp_from_forces(F_m, pos, probes4, create_graph=False).detach().numpy().reshape(4, -1)
    ref4 = (probes4.numpy().reshape(4, -1)) @ Hp_r.T
    est = np.mean(np.sum((hv4 - ref4) ** 2, axis=1)) / (9 * Np * Np)
    print(f"  (2.2) k = 4 probes on the live model: {est:.4e}  against the exact ‖ΔH‖²/(9N²) = {np.sum((Hm - Hp_r) ** 2)/(9*Np*Np):.4e}  (one draw; unbiased over draws, §2)")
    # the third-order graph (4.2)
    params = list(model.parameters())
    for q in params: q.requires_grad_(True)
    try:
        F_g, pos_g = ef()
        hv = hvp_mod.hvp_from_forces(F_g, pos_g, probes[:1], create_graph=True)
        gpar = torch.autograd.grad(hv.pow(2).sum(), params, allow_unused=True)
        nz = sum(g is not None and float(g.abs().max()) > 0 for g in gpar)
        print(f"  (4.2) the HVP carries a graph to θ: {nz} of {len(params)} parameter tensors receive a non-zero gradient")
    finally:
        for q in params: q.requires_grad_(False)
except Exception as exc:                                                 # noqa: BLE001
    print("MACE not available here -- timing skipped:", str(exc)[:160])

/home/ubuntu/anaconda3/envs/openqha/lib/python3.11/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


/mnt/c/Users/10704/Documents/01_Free-Energy-alchemical/openQHA-Hessian/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


MACE-OFF23_medium, N = 10 (3N = 30), CPU wall (best of 3):
  energy + forces with graph           158.4 ms   (= 1 unit)
   1 HVP on that graph                 105.8 ms    0.67 units    0.67 per probe
   4 HVP on that graph                 420.9 ms    2.66 units    0.66 per probe
   8 HVP on that graph                 838.9 ms    5.29 units    0.66 per probe
  30 HVP on that graph                3166.2 ms   19.98 units    0.67 per probe
  get_hessian (vmap over 3N = 30)     3458.4 ms   21.83 units


  3N unit-probe HVPs == get_hessian to 0.0e+00 eV/Å²;  stored fixture Hessian to 1.4e-14


  (2.2) k = 4 probes on the live model: 2.2025e-03  against the exact ‖ΔH‖²/(9N²) = 2.5910e-03  (one draw; unbiased over draws, §2)


  (4.2) the HVP carries a graph to θ: 29 of 29 parameter tensors receive a non-zero gradient


## 5. The total objective: basin frames labelled, Replay frames unlabelled

The fine-tune Dataset's `train` split holds basin frames only, each with $E$, $F$
and $H$; the Replay's SPICE frames (T05 §6; one drawn file, `config_weight` per frame) carry
$E$ and $F$ only and sit in the same shuffled training set as mace's pretraining head. Let
$\mathcal B$ be a batch of graphs and $\mathcal B_H\subseteq\mathcal B$ those with a Label. The objective is
$$\mathcal L(\theta)=w_E\,\mathcal L_E(\mathcal B)+w_F\,\mathcal L_F(\mathcal B)+w_H\,\frac1{|\mathcal B_H|}\sum_{n\in\mathcal B_H}\hat{\mathcal L}_{H,n}^{(K)}(\theta) ,\tag{5.1}$$
with mace's own per-graph E/F terms (each frame's `config_weight` multiplying its E/F
terms — the replay weight knob), and the Hessian term averaged over the labelled graphs only.
Four consequences follow from what was derived above:

1. *Masking is exact.* By §2 each $\hat{\mathcal L}_{H,n}$ is an unbiased estimate of $\|\Delta H_n\|_F^2/(9N_n^2)$, so
   the masked mean is an unbiased estimate of the masked population loss; graphs without a
   Label (the Replay frames) contribute $0$ and no gradient through the Hessian term — they
   still drive E and F, which is their job. The same estimator with $k=4$ probes *fixed per
   frame* (seeded from the Label's bytes) is the validation term: unbiased for the
   same reason, and reproducible because the draw does not change between epochs.
2. *Batching is free.* A batch is a block-diagonal graph, so $-\nabla_x\sum_nF_n\!\cdot\!v_{n,j}$ returns
   every graph's own $H_nv_{n,j}$ in one backward (PHL's "single HVP call" with the
   concatenated $v$): the Hessian term costs $K$ backward passes per batch regardless of $|\mathcal B|$;
   an unlabelled graph gets a zero probe.
3. *The weight $w_H$ has no universal value.* $\mathcal L_H$ is in eV$^2$ Å$^{-4}$ and its size depends on
   the molecules and on the base model's state. PHL's $w_H/w_F=0.09/0.30$ was tuned for ANI in
   Hartree from scratch and would put the Hessian term at $10\times$ the force term on our
   fixture (T05 §4). The rule that does transfer: at epoch 0, $w_H\hat{\mathcal L}_H=w_F\mathcal L_F$ on the
   base model — `05_train`'s default (`--hessian-weight balance`, `run.hessian_weight_balance`
   over the run's own train file with the full matrix (2.1) per Hessian frame); no scan.
4. *Two stages.* When mace switches to Stage Two the loss is rebuilt from the `swa_*` weights;
   $w_H^{\rm II}=w_H\,w_F^{\rm II}/w_F$ keeps the Hessian share unchanged (`run.stage_two_weights`, T05 §6).

The cell checks 1–2 on a torch toy potential with a three-graph batch (two labelled, one
not — a Replay-like graph), with the probes and reference matvecs from `phl.make_probes` as the
code runs them, and the gradient (4.2) by finite differences through the masked batch and the
third-order graph.

In [6]:
class ToyPotential(torch.nn.Module):
    '''E(x) = Σ_{i<j} f_θ(r_ij) + Σ_i g_θ(|x_i - x_com|²): pair MLP + a confining term, parameters θ.'''
    def __init__(self):
        super().__init__()
        self.pair = torch.nn.Sequential(torch.nn.Linear(1, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1))
        self.conf = torch.nn.Parameter(torch.tensor(0.05))
    def forward(self, x, batch=None):
        n = x.shape[0]
        idx = torch.zeros(n, dtype=torch.long) if batch is None else batch
        same = (idx[:, None] == idx[None, :]).triu(1)
        diff = x[:, None, :] - x[None, :, :]
        d = torch.sqrt((diff ** 2).sum(-1)[same] + 1e-12).unsqueeze(-1)
        e_pair = self.pair(d).sum()
        com = x.mean(0, keepdim=True) if batch is None else torch.stack([x[idx == g].mean(0) for g in idx.unique()])[idx]
        return e_pair + self.conf * ((x - com) ** 2).sum()

torch.manual_seed(0); toy = ToyPotential()
k = 3
# three graphs: propanal (labelled), the 2-methyloxirane basin (labelled), its displaced frame 2 (E/F only: a held-out or
# Replay-like graph)
graphs = [(xp, mp, Hp_r), (X[0], mm, H_r[0]), (X[2], mm, None)]
def hessian_term(model, graphs, probes_by_graph):
    '''(5.1)'s Hessian term on a block-diagonal batch: one forward, one force backward, k HVP backwards.'''
    xs = torch.tensor(np.concatenate([g[0] for g in graphs]), requires_grad=True)
    idx = torch.cat([torch.full((len(g[0]),), i, dtype=torch.long) for i, g in enumerate(graphs)])
    E = model(xs, idx); F = -torch.autograd.grad(E, xs, create_graph=True)[0]
    offs = np.cumsum([0] + [len(g[0]) for g in graphs])
    total, n_lab = 0.0, 0
    hv_all = []
    for j in range(k):
        probe = torch.zeros_like(xs)
        for n, g in enumerate(graphs):
            if g[2] is not None:
                probe[offs[n]:offs[n+1]] = torch.tensor(probes_by_graph[n][0][j].reshape(-1, 3))
        hv_all.append(torch.autograd.grad(-(F * probe).sum(), xs, create_graph=True)[0])
    for n, g in enumerate(graphs):
        if g[2] is None:
            continue
        vt, r, info = probes_by_graph[n]
        acc = 0.0
        for j in range(k):
            hv = hv_all[j][offs[n]:offs[n+1]].reshape(-1)
            rho = hv - torch.tensor(r[j])                                       # (2.2): H_θ v_j − H_r v_j
            acc = acc + rho.pow(2).sum()
        total = total + acc / info["denominator"]; n_lab += 1
    return total / n_lab
probes_by_graph = {0: phl.make_probes(Hp_r, mode="rademacher", k=k, rng=np.random.default_rng(10)),
                   1: phl.make_probes(H_r[0], mode="rademacher", k=k, rng=np.random.default_rng(11))}
i0 = probes_by_graph[0][2]
print(f"phl.make_probes: the probe the model sees is the raw ±1 draw {bool(set(np.unique(probes_by_graph[0][0]).tolist()) <= {-1.0, 1.0})}, "
      f"r_j = H_r v_j {np.allclose(probes_by_graph[0][1], probes_by_graph[0][0] @ Hp_r.T)}, denominator 9N²k = {i0['denominator']} ({9 * Np * Np * k})")
L_batch = hessian_term(toy, graphs, probes_by_graph)
L_sep = [hessian_term(toy, [g], {0: probes_by_graph[n]}) for n, g in enumerate(graphs) if g[2] is not None]
print(f"(5.1) masked batch of 3 (2 labelled): {L_batch.item():.12e}   mean of the two alone: {np.mean([l.item() for l in L_sep]):.12e}   diff {abs(L_batch.item() - np.mean([l.item() for l in L_sep])):.1e}")
p = toy.pair[0].weight; g_auto = torch.autograd.grad(L_batch, p)[0][3, 0].item()
eps = 1e-4
with torch.no_grad(): p[3, 0] += eps
Lp = hessian_term(toy, graphs, probes_by_graph).item()
with torch.no_grad(): p[3, 0] -= 2 * eps
Lm = hessian_term(toy, graphs, probes_by_graph).item()
with torch.no_grad(): p[3, 0] += eps
print(f"(4.2) ∂L/∂θ autograd {g_auto:+.8e}   central FD {(Lp - Lm) / (2 * eps):+.8e}   |diff| {abs(g_auto - (Lp - Lm) / (2 * eps)):.1e}")

phl.make_probes: the probe the model sees is the raw ±1 draw True, r_j = H_r v_j True, denominator 9N²k = 2700 (2700)
(5.1) masked batch of 3 (2 labelled): 5.125345830289e+01   mean of the two alone: 5.125345830289e+01   diff 0.0e+00
(4.2) ∂L/∂θ autograd +1.16706556e-01   central FD +1.16706557e-01   |diff| 6.6e-10


## 6. Algorithm listings — complete, as the code stands (PHL verbatim)

Notation as above; `←` assignment; every step names the function that implements it
(`openqha_hessian/phl.py`, `hvp.py`, `phl_loss.py`, `run.py`, `judge.py`; `openqha/data/dataset.py`).

```
Algorithm 0  dataset(frames, train_generators = (basin,))            -- who trains on what (dataset.build)
  1  for every labelled frame:  if generator ∉ train_generators:  split ← test, held_out_generator ← yes   # above the draw
     else: the by-frame draw (90 / 5 / 5) or the previous index's decision
  2  Replay ← s0_spice_pt_draw.py --n N --seed S --weight W          # one permutation of SPICE's train split; the first N eligible
     frames (no forgetting-draw molecule, no in_distribution molecule); config_weight = W; <stem>.valid.extxyz from the end
     (round 1's production draw: N = 30000, one seed, written twice at W = 1 and 10 -- the same frame set)
  3  N_TRAIN_HESSIAN ← |{train frames with a Label}|   # the Dataset Record; its S0 ladder field REPLAY_R4_FRAMES = 4 × N_TRAIN_HESSIAN stays as labelled history
  Output  train / valid (basin frames), test (drawn + held out), the Replay file and its companion
```

```
Algorithm 1  frame_constants(H_r)                                  -- once per labelled frame, no autograd (phl_loss.FrameConstants)
  Input   H_r [3N,3N] the raw Cartesian Label, as stored
  1  ν ← 9 N²                                                             # PHL's (3N)² normalisation
  Output  (H_r, ν)                                                        # nothing is diagonalised, projected or hashed (the validation probes are STORED)
```

```
Algorithm 2  probes(frame constants, mode, K, rng)                  -- per frame per batch, no autograd (phl.make_probes)
  1  if mode = rademacher:  v_j ~ Uniform{−1,+1}^{3N}, j = 1..K          denominator ← ν · K      (the smaller variance, 2.3; a choice, not the default)
     if mode = gaussian:    v_j ~ N(0, I),               j = 1..K          denominator ← ν · K      (PHL's Algorithm 1; the default)
     if mode = cartesian:   v_j ← e_j,                   j = 1..3N        denominator ← ν          (exact: = get_hessian, §2 / §4)
     the draw comes from rng in training (mace's seed); at validation it is the frame's STORED set, read from the batch
  2  r_j ← H_r v_j                               # the reference side: a matvec on the Label (PHL's bmm(H_ref, v))
  Output  {v_j}, {r_j}, denominator              # the probe the model sees is the draw itself
```

```
Algorithm 3  training_step(batch 𝓑, θ, K, w_E, w_F, w_H)          -- one optimiser step (phl_loss.forward inside mace's train loop)
  1  for n in 𝓑 with a Label:  ({v_{n,j}}, {r_{n,j}}, den_n) ← Algorithm 2      # Replay frames and any unlabelled graph: v_{n,j} ← 0
  2  E_θ, F_θ ← model(𝓑, training=True)          # forces with create_graph=True; batch.positions is the leaf
  3  for j = 1..K:
  4      g_j ← autograd.grad( −Σ_n F_{θ,n} · v_{n,j},  positions,  create_graph=True )   # = [H_{θ,n} v_{n,j}]_n, one backward for the whole batch (4.1)  (hvp.hvp_from_forces)
  5  for n in 𝓑_H:  L̂_n ← Σ_j ‖ g_{n,j} − r_{n,j} ‖² / den_n                     # (2.2), per-graph slice by ptr  (phl_loss.hvp_error)
  6  L_H ← (1/|𝓑_H|) Σ_{n ∈ 𝓑_H} L̂_n                                               # masked mean (5.1); 0 · Σ F if 𝓑_H = ∅
  7  L ← w_E L_E(𝓑) + w_F L_F(𝓑) + w_H L_H                                       # mace's E/F terms, each frame × its config_weight
  8  θ ← optimizer.step( ∂L/∂θ )                 # the parameter backward runs through the third-order graph of step 4 (4.2)
  (multihead: the Replay's frames are in the same batch and pay E/F only -- commit C keeps this loss there;
   Stage Two at 3/4 of the epochs rebuilds it from the swa_* weights, w_H^II = w_H w_F^II / w_F -- run.control_settings;
   w_H itself = w_F L_F / L_H on the base over the train file, the driver's default rule -- run.hessian_weight_balance)
```

```
Algorithm 4  evaluate(validation set, θ)                            -- the fixed-probe estimator (phl_loss in eval mode)
  1  loss.eval();  for each batch:  E_θ, F_θ ← model(𝓑, training=True)   # the force graph kept: wants_force_graph_at_eval (commit C)
  2  for each labelled frame:  ({v_j}, {r_j}, den) ← Algorithm 2 with the frame's STORED probes, K = 4 gaussian   # the same 4 every epoch
  3      L̂_n ← Σ_j ‖ g_j − r_j ‖² / den                                    # under enable_grad (torchmetrics runs update under no_grad)
  4  accumulate w_E L_E, w_F L_F, mean_n L̂_n; total ← their sum            # what ReduceLROnPlateau, the best checkpoint, Stage Two read
  5  eval_summary() → valid_energy_term, valid_forces_term, valid_hessian_term → results/*.txt → the Record's three curves (run.parse_results)
  (the full matrix is never asked for: wants_hessian_at_eval = False; it is the judge's tool, Algorithm 5)
```

```
Algorithm 5  judge(test split, θ, base)                              -- the ruler (judge.run)
  1  for each labelled frame of test, for engine ∈ {θ, base}:
  2      E, F ← get_potential_energy / get_forces:  |ΔE|/N, F RMSE                (every frame, held out or drawn)
  3      if the frame has a Label Hessian:  H ← get_hessian (= the 3N unit probes, §4);
             ‖H − H_r‖²_F/(9N²)  (2.1, the target: judge's loss_cartesian);
             hessian_compare(H, H_r, M, x): element MAE, and -- the standard vibrational analysis, mass-weighting + Eckart
             projection, an evaluation step -- frequency MAE all / low, mode overlap
  4  aggregate: per distribution and per class over the Hessian frames (judge.aggregate)
  5  reference bins: (distribution, rms_bin ∈ {0, <0.08, <0.15, ≥0.15 Å}) over ALL frames: H over the Hessian frames, E/F over all (judge.aggregate_displacement; ρ_k of 3.3)
  6  thermochemistry: MODEL_ERROR_S_REF per pinned molecule, READ from the engine's own msRRHO Records under its tag (--thermo-tag; never recomputed)
  7  forgetting: E/F RMSE on the fixed SPICE test draw, ratio to base
  8  MD ramp (reference, only when asked for): from each model's own minimum, Langevin 1 fs, 5 K, +5 K every 5 ps, until a pair's 50-step mean distance
     leaves [0.75, 1.5] × equilibrium or 600 K; FAIL_T_K per molecule, engine and base (judge.ramp_one / md_ramp)
  9  verdict lines with GATE (the gate rows):   yes -- held_out_hessian_cartesian (the matrix itself: engine ‖ΔH‖²/(9N²) / base − 1 ≤ 0),
                                                     in_distribution_degradation (≤ 15 %), forgetting (≤ 1.15×)
                                           no  -- held_out_low_mode_mae_cm (≤ 8.5), model_error_s_ref (≤ 0.2 cal/mol/K) -- computed from the matrix --
                                                  held_out_generator_hessian_vs_base, md_ramp_K (the ramp only with --ramp)
     VERDICT ← REPORTED while the gate is closed (the default); with --gate: PASS iff every gate row is PASS or '-'
     (judge.verdict_of); a reference row never moves it
     calibration (gate open): base vs base reads exactly 0 on the Hessian gate; the 0.81x Hessian fails it and the verdict
```

## 7. Acceptance, the code map, and how T05 uses each piece

| id | statement | cell | result here |
|---|---|---|---|
| S1 | E/F training cannot fix the curvature at or between samples (Prop. 1) | §1 | $\tilde E$ built with $E,F$ equal, $H$ arbitrary |
| S2 | a displaced force fixes one column-direction of the path-averaged Hessian (1.1) | §1 | trapezoid to 1–19 %; basin $H$ alone off by 20–90 %; $H_r$ changes 15–22 % over a 298 K frame |
| S3 | PHL's estimator (2.2) is unbiased for Rademacher and Gaussian probes; the Rademacher variance is (2.3) and the smaller; $3N$ unit probes are exact | §2 | within a few s.e.; variance $=$ (2.3); exact |
| S4 | Weyl: every $|\delta\lambda_i|\le\|\Delta H\|_F/m_{\min}$ (2.4) — the target is sufficient for every frequency, loosely | §2 | chain holds; loose by orders of magnitude on the soft modes |
| S5 | the held-out generator's reading $\rho_k$ (3.3) and $\Delta\mathcal T[d]$ (3.2) are measurable from the frame fixture before any training | §3 | table |
| S6 | one HVP $\approx1\,u$; $3N$ HVPs $=$ `get_hessian`; (2.2) on the live model | §4 | 1.0–1.2 $u$ per probe; identical to 0 |
| S7 | masking is exact and the gradient runs through the third-order graph (4.2, 5.1) | §5 | batch $=$ mean of labelled; FD to 1e-8 |

**Code map.** Algorithm 0: `dataset.build(train_generators=)`, `dataset.TRAIN_GENERATORS`,
`dataset.REPLAY_N_FRAMES` / `REPLAY_SEED` / `REPLAY_WEIGHTS` (the production draw; the S0
ladder's `REPLAY_PER_HESSIAN_FRAME_R4` / `REPLAY_CONFIG_WEIGHT_R4` stay as history),
`scripts/tooling/s0_spice_pt_draw.py`
(`scan_frames`, `classify`, `draw`, `check_disjoint`). Algorithm 1: `phl_loss.FrameConstants`,
Algorithm 2: `phl.make_probes`, `phl.estimator_from_products`,
`phl.estimator_variance`, `FrameConstants.probes` (fixed at validation); the full matrix
`phl.cartesian_loss_full`. Algorithm 3: `hvp.hvp_from_forces`,
`phl_loss.WeightedEnergyForcesHessianLoss.hvp_error` (the `nn.Module` mace's `train()` calls through
the fork's external-loss hook; commit C keeps it in multihead mode), `run.mace_argv` /
`run.control_settings` / `run.stage_two_weights` for the flags, `run.hessian_weight_balance` for
$w_H$ (`05_train.py --hessian-weight balance`, the default). Algorithm 4: `wants_force_graph_at_eval`,
`eval_summary`, `run.parse_results` / `run.validation_curves` / `run.curve_moved`. Algorithm 5:
`judge.frame_rows` / `aggregate` / `aggregate_displacement` / `thermochemistry` / `forgetting` /
`ramp_one` / `md_ramp` / `verdict` / `verdict_of`, with `hessian_compare.compare_hessians` (the
standard analysis for the frequency rows); `06_judge.py --gate --ramp --ramp-max-K --thermo-tag`.
*(The switch was removed: `phl.make_probes(H_r, mode, k, rng)` and
`phl.estimator_variance(H_theta, H_r, k)` have one target and no `metric` argument. The projected
branch was the superseded T03 design; what is left of it is the judge's frequency analysis.)*

**How T05 uses each section.** T05 §1 (the base model's E/F/H numbers) is what S1 predicts must be
there. T05 §2 (PHL's loss verbatim) is §2 here: (2.1)–(2.3), the exactness of the unit probes, and
what the target bounds (2.4). T05 §3 (basin frames only; the held-out generator) is Proposition 1
plus (3.1)–(3.3) read as the held-out generator's reference rows, not as a reason to label more.
T05 §4 ($w_H$ measured) is §5 item 3. T05 §5 (the Replay unlabelled, fixed-probe validation, two
stages) is (5.1), items 1 and 4, and Algorithm 4. T05 §6 (gate and reference rows) is Algorithm 5.
T05 §7's recipe is Algorithms 0–5 with their flags.

References: PHL — *Projected Hessian Learning (PHL): Fast Curvature Supervision for Accurate MLIPs*
(eq. 6, Algorithm 1; reference implementation `PHL-main/PHL_training.ipynb`); Hutchinson, *Commun.
Stat. Simul.* 18, 1059 (1989); Baur & Strassen, *Theor. Comput. Sci.* 22, 317 (1983) (reverse-mode
cost); Weyl, *Math. Ann.* 71, 441 (1912) (eigenvalue perturbation bound); Rodriguez, Smith,
Mendoza-Cortes, *JCTC* 2025, 10.1021/acs.jctc.5c00402 (stationary-only Hessian training); Koker et
al., *PFT*, arXiv:2601.07742; Burger et al., *HIP* (§5: frequencies after mass-weighting and Eckart
projection, an evaluation step); Louck & Galbraith, *Rev. Mod. Phys.* 48, 69 (1976) and
gaussian.com/vib for that analysis; T03 (the superseded projected design, kept for the record) and T05
of this folder.